# 03 · Delta Lake Operations

This notebook uses a small inventory table to demonstrate the Delta Lake operations that make lakehouse tables reliable and auditable.

### Skills demonstrated
- Creating and appending to Delta tables
- SQL UPDATE
- MERGE logic
- Delta transaction history
- Time travel with VERSION AS OF
- Schema enforcement


## 1. Create the inventory Delta table


In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS portfolio_lab;
DROP TABLE IF EXISTS portfolio_lab.inventory;


In [ ]:
inventory = spark.createDataFrame(
    [
        ("SKU001", "Wireless Mouse", "Accessories", 100, 24.99),
        ("SKU002", "Mechanical Keyboard", "Accessories", 60, 79.99),
        ("SKU003", "USB-C Hub", "Accessories", 80, 39.99),
        ("SKU004", "Webcam", "Peripherals", 45, 59.99),
        ("SKU005", "Laptop Stand", "Peripherals", 70, 34.99),
        ("SKU006", "USB Headset", "Peripherals", 55, 49.99),
    ],
    ["sku", "product_name", "category", "quantity", "unit_price"],
)

(
    inventory.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("portfolio_lab.inventory")
)

display(spark.table("portfolio_lab.inventory"))


## 2. Append new products


In [ ]:
new_products = spark.createDataFrame(
    [
        ("SKU007", "Portable SSD", "Storage", 35, 109.99),
        ("SKU008", "Flash Drive", "Storage", 120, 19.99),
    ],
    schema=inventory.schema,
)

(
    new_products.write
    .format("delta")
    .mode("append")
    .saveAsTable("portfolio_lab.inventory")
)

print(f"Current row count: {spark.table('portfolio_lab.inventory').count()}")


## 3. Update existing records

A targeted SQL update increases prices in one category by 5%.


In [ ]:
%sql
UPDATE portfolio_lab.inventory
SET unit_price = ROUND(unit_price * 1.05, 2)
WHERE category = 'Peripherals';


In [ ]:
%sql
SELECT *
FROM portfolio_lab.inventory
ORDER BY sku;


## 4. MERGE updates and inserts

MERGE implements an upsert: matching SKUs are updated while new SKUs are inserted.


In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW inventory_updates AS
SELECT
    'SKU001' AS sku,
    'Wireless Mouse' AS product_name,
    'Accessories' AS category,
    95 AS quantity,
    CAST(22.99 AS DOUBLE) AS unit_price

UNION ALL

SELECT
    'SKU009' AS sku,
    '4K Monitor' AS product_name,
    'Displays' AS category,
    25 AS quantity,
    CAST(299.99 AS DOUBLE) AS unit_price;


In [ ]:
%sql
MERGE INTO portfolio_lab.inventory AS target
USING inventory_updates AS source
ON target.sku = source.sku

WHEN MATCHED THEN
    UPDATE SET
        target.quantity = source.quantity,
        target.unit_price = source.unit_price

WHEN NOT MATCHED THEN
    INSERT *;


In [ ]:
%sql
SELECT *
FROM portfolio_lab.inventory
ORDER BY sku;


## 5. Inspect table history

Every write operation creates a new Delta table version

In [ ]:
%sql
DESCRIBE HISTORY portfolio_lab.inventory


### Query the original version



In [ ]:
%sql
SELECT *
FROM portfolio_lab.inventory VERSION AS OF 0
ORDER BY sku;


### Compare original and current prices



In [ ]:
%sql
SELECT
    current.sku,
    current.product_name,
    original.unit_price AS original_price,
    current.unit_price AS current_price,
    ROUND(current.unit_price - original.unit_price, 2) AS price_change
FROM portfolio_lab.inventory AS current
JOIN portfolio_lab.inventory VERSION AS OF 0 AS original
    ON current.sku = original.sku
ORDER BY current.sku;


## 6. Demonstrate schema enforcement

The following DataFrame intentionally does not match the inventory schema. The append is wrapped in try/except so the notebook can continue after Delta rejects the write.


In [ ]:
invalid_inventory = spark.createDataFrame(
    [("SKU999", 10)],
    ["sku", "quantity"],
)

try:
    (
        invalid_inventory.write
        .format("delta")
        .mode("append")
        .saveAsTable("portfolio_lab.inventory")
    )
    print("Unexpected result: the invalid write succeeded.")
except Exception as exc:
    print("Expected failure: Delta rejected the mismatched schema.")
    print(str(exc).splitlines()[0])


## Key takeaways

- Delta Lake records table changes as versions.
- MERGE is useful when a source contains both updates and new records.
- Time travel makes before and after comparisons straightforward.
- Schema enforcement protects tables from incompatible writes.
